In [ ]:
#@title Captioner for Colab { display-mode: "form" }
import html
import os
import pathlib
import re
import shutil
import subprocess
import threading
import time
from IPython.display import HTML, display

REPO_URL = "https://github.com/GodL-x-SouL/Captioner-for-Colab.git"
APP_DIR = pathlib.Path("/content/Captioner-for-Colab")
APP_URL = "http://127.0.0.1:7860"

logs = []
tunnel_url = ""
status = "Preparing runtime"
captioner_proc = None
tunnel_proc = None
display_handle = display(HTML(""), display_id=True)

def add_log(line):
    line = str(line).rstrip()
    if line:
        logs.append(line)
    del logs[:-220]

def panel():
    safe_logs = html.escape("\n".join(logs[-160:]))
    safe_status = html.escape(status)
    safe_tunnel = html.escape(tunnel_url or "Waiting for Cloudflare Tunnel...")
    open_link = f'<a class="open" href="{safe_tunnel}" target="_blank">Open Captioner</a>' if tunnel_url else '<span class="open muted">Starting...</span>'
    return f'''
    <style>
      .cap-wrap{{width:98%;min-height:520px;margin:10px auto 0;padding:22px;border:1px solid #c6b78a;border-radius:8px;background:#f7f2e8;color:#233126;font-family:Inter,Arial,sans-serif;box-shadow:0 18px 55px rgba(41,35,20,.18)}}
      .cap-head{{display:flex;align-items:flex-start;justify-content:space-between;gap:18px;border-bottom:1px solid #d8cda9;padding-bottom:16px;margin-bottom:16px}}
      .cap-title{{font-family:Georgia,serif;font-size:28px;letter-spacing:.2px;color:#17251c;line-height:1.1}}
      .cap-sub{{font-size:12px;color:#6f674e;margin-top:6px;letter-spacing:.08em;text-transform:uppercase}}
      .open{{display:inline-flex;align-items:center;justify-content:center;min-width:150px;padding:9px 14px;border:1px solid #173b2c;border-radius:4px;background:#173b2c;color:#fbf7ee;text-decoration:none;font-size:12px;font-weight:600;letter-spacing:.08em;text-transform:uppercase}}
      .open.muted{{background:#d9cfb0;color:#6b624b;border-color:#d0c49f}}
      .cap-grid{{display:grid;grid-template-columns:1fr;gap:14px}}
      .cap-card{{border:1px solid #d8cda9;border-radius:6px;background:#fffaf0;padding:14px}}
      .label{{font-size:10px;color:#776c4d;letter-spacing:.12em;text-transform:uppercase;margin-bottom:8px}}
      .status{{font-family:Georgia,serif;font-size:19px;color:#173b2c}}
      .urlbox{{font-family:ui-monospace,SFMono-Regular,Consolas,monospace;font-size:13px;border:1px solid #c9ba8d;border-radius:4px;background:#f1ead8;padding:12px;word-break:break-all;color:#2f3c2f}}
      .logs{{height:260px;overflow:auto;white-space:pre-wrap;font-family:ui-monospace,SFMono-Regular,Consolas,monospace;font-size:11px;line-height:1.55;color:#314033;background:#11170f;border:1px solid #27311f;border-radius:6px;padding:12px}}
      .logs::-webkit-scrollbar{{width:8px}}.logs::-webkit-scrollbar-track{{background:#11170f}}.logs::-webkit-scrollbar-thumb{{background:#b9a56b;border-radius:8px;border:2px solid #11170f}}
      @media(max-width:700px){{.cap-head{{flex-direction:column}}.cap-title{{font-size:23px}}.cap-wrap{{width:99%;padding:16px}}}}
    </style>
    <div class="cap-wrap">
      <div class="cap-head">
        <div><div class="cap-title">Captioner for Colab</div><div class="cap-sub">Private runtime launcher</div></div>
        {open_link}
      </div>
      <div class="cap-grid">
        <div class="cap-card"><div class="label">Status</div><div class="status">{safe_status}</div></div>
        <div class="cap-card"><div class="label">Tunnel URL</div><div class="urlbox">{safe_tunnel}</div></div>
        <div class="cap-card"><div class="label">Logs</div><div class="logs">{safe_logs}</div></div>
      </div>
    </div>'''

def refresh():
    display_handle.update(HTML(panel()))

def run_quiet(cmd, cwd=None, label=None, timeout=None):
    global status
    if label:
        status = label
        add_log(label)
        refresh()
    proc = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in proc.stdout:
            add_log(line)
            if len(logs) % 4 == 0:
                refresh()
        code = proc.wait(timeout=timeout)
    except Exception:
        proc.kill()
        raise
    if code != 0:
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")
    refresh()

def stream_process(proc, label, url_pattern=None):
    global tunnel_url
    for line in proc.stdout:
        add_log(f"[{label}] {line}")
        if url_pattern and not tunnel_url:
            match = re.search(url_pattern, line)
            if match:
                tunnel_url = match.group(0)
        refresh()

try:
    refresh()
    if not APP_DIR.exists():
        run_quiet(["git", "clone", "--depth", "1", REPO_URL, str(APP_DIR)], label="Cloning repository")
    else:
        run_quiet(["git", "-C", str(APP_DIR), "pull", "--ff-only"], label="Updating repository")

    run_quiet(["apt-get", "update", "-qq"], label="Preparing system packages")
    run_quiet(["apt-get", "install", "-y", "-qq", "aria2"], label="Installing aria2")
    run_quiet(["python", "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=str(APP_DIR), label="Installing Python requirements")

    if not shutil.which("cloudflared"):
        run_quiet(["wget", "-q", "-O", "/usr/local/bin/cloudflared", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], label="Installing Cloudflare Tunnel")
        run_quiet(["chmod", "+x", "/usr/local/bin/cloudflared"], label="Finalizing Cloudflare Tunnel")

    status = "Starting captioner"
    refresh()
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    captioner_proc = subprocess.Popen(["python", "captioner.py"], cwd=str(APP_DIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    threading.Thread(target=stream_process, args=(captioner_proc, "captioner"), daemon=True).start()

    deadline = time.time() + 45
    while time.time() < deadline:
        if any("opening http://127.0.0.1:7860" in line.lower() for line in logs[-40:]):
            break
        time.sleep(1)

    status = "Opening Cloudflare Tunnel"
    refresh()
    tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", APP_URL, "--no-autoupdate"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    threading.Thread(target=stream_process, args=(tunnel_proc, "tunnel", r"https://[-a-zA-Z0-9.]+\.trycloudflare\.com"), daemon=True).start()

    deadline = time.time() + 120
    while time.time() < deadline and not tunnel_url:
        time.sleep(1)
        refresh()

    status = "Ready" if tunnel_url else "Tunnel still starting"
    refresh()

    while True:
        if captioner_proc.poll() is not None:
            status = "Captioner stopped"
            refresh()
            break
        if tunnel_proc.poll() is not None:
            status = "Tunnel stopped"
            refresh()
            break
        time.sleep(2)
        refresh()
except KeyboardInterrupt:
    status = "Stopped by user"
    for proc in (captioner_proc, tunnel_proc):
        if proc and proc.poll() is None:
            proc.terminate()
    refresh()
except Exception as exc:
    status = "Setup failed"
    add_log(str(exc))
    refresh()
